In [1]:
import csv
import json
import re
import warnings
from pathlib import Path

import torch
import torch._dynamo  # pre-initialize before transformers to avoid circular import
import spacy
from dotenv import load_dotenv
from transformers.pipelines import TokenClassificationPipeline
from deepmultilingualpunctuation import PunctuationModel

load_dotenv("../.env")

# Patch grouped_entities compatibility (safe to apply multiple times)
if not getattr(TokenClassificationPipeline, "_grouped_entities_patched", False):
    _orig_sanitize = TokenClassificationPipeline._sanitize_parameters

    def _patched_sanitize(self, **kwargs):
        if "grouped_entities" in kwargs:
            kwargs["aggregation_strategy"] = "simple" if kwargs.pop("grouped_entities") else "none"
        return _orig_sanitize(self, **kwargs)

    TokenClassificationPipeline._sanitize_parameters = _patched_sanitize
    TokenClassificationPipeline._grouped_entities_patched = True

RAW_DIR = Path("../data/raw/daicwoz")
OUT_FILE = Path("../data/processed/daicwoz_finetune.jsonl")
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

EXCLUDE_SESSIONS = {451, 458, 480}

SYSTEM_PROMPT = """\
You are a virtual clinical interviewer. Conduct a structured, empathetic \
mental health interview by asking open-ended questions and responds naturally \
to what the participant shares."""

NOISE_PATTERN = re.compile(r"<[^>]+>|(?<!\w)xxx(?!\w)", re.IGNORECASE)
TAG_PAREN_PATTERN = re.compile(r"^\w+\s+\((.+)\)$")
SENTENCE_START_PATTERN = re.compile(r"([.!?]\s+)([a-z])")
PRONOUN_I_PATTERN = re.compile(r"\bi\b")
ELLIE_PATTERN = re.compile(r"\bellie\b", re.IGNORECASE)
ABBREV_PATTERN = re.compile(r"\b([a-z])(_[a-z])+\b")

def expand_abbreviations(text: str) -> str:
    return ABBREV_PATTERN.sub(lambda m: m.group(0).replace("_", "").upper(), text)

def clean(text: str) -> str:
    m = TAG_PAREN_PATTERN.match(text.strip())
    if m:
        text = m.group(1)
    text = NOISE_PATTERN.sub("", text)
    text = expand_abbreviations(text)
    return " ".join(text.split()).strip()

def capitalize_sentences(text: str) -> str:
    text = text[:1].upper() + text[1:] if text else text
    return SENTENCE_START_PATTERN.sub(lambda m: m.group(1) + m.group(2).upper(), text)

def fix_pronoun_i(text: str) -> str:
    return PRONOUN_I_PATTERN.sub("I", text)

def fix_ellie(text: str) -> str:
    return ELLIE_PATTERN.sub("Ellie", text)

def capitalize_proper_nouns(text: str, nlp) -> str:
    doc = nlp(text)
    return "".join(
        (token.text if token.text.isupper() else token.text.capitalize()) + token.whitespace_
        if token.pos_ == "PROPN"
        else token.text_with_ws
        for token in doc
    )

def load_conversation(csv_path: Path) -> list[dict]:
    rows = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            speaker = row["speaker"].strip()
            value = clean(row["value"])
            if not value:
                continue
            role = "assistant" if speaker == "Ellie" else "user"
            if rows and rows[-1]["role"] == role:
                rows[-1]["content"] += " " + value
            else:
                rows.append({"role": role, "content": value})
    return rows

# --- Load models ---
print("Loading punctuation model...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    punct_model = PunctuationModel()

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

# --- Load all conversations ---
print("Loading conversations...")
all_conversations = []
for csv_path in sorted(RAW_DIR.glob("*_TRANSCRIPT.csv")):
    session_id = int(csv_path.name.split("_")[0])
    if session_id in EXCLUDE_SESSIONS:
        continue
    turns = load_conversation(csv_path)
    if len(turns) >= 2:
        all_conversations.append(turns)

print(f"Loaded {len(all_conversations)} conversations")

# --- Restore punctuation & apply capitalization fixes ---
print("Restoring punctuation and fixing capitalization...")
coords, texts = [], []
for c_idx, turns in enumerate(all_conversations):
    for t_idx, turn in enumerate(turns):
        coords.append((c_idx, t_idx))
        texts.append(turn["content"])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    restored_texts = [punct_model.restore_punctuation(t) for t in texts]

for (c_idx, t_idx), restored in zip(coords, restored_texts):
    text = capitalize_sentences(restored)
    text = fix_pronoun_i(text)
    text = fix_ellie(text)
    text = capitalize_proper_nouns(text, nlp)
    all_conversations[c_idx][t_idx]["content"] = text

print("Done.")

# --- Write JSONL ---
records_written = 0
with OUT_FILE.open("w", encoding="utf-8") as out_f:
    for turns in all_conversations:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + turns
        out_f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        records_written += 1

print(f"Wrote {records_written} conversations → {OUT_FILE}")
print(f"\nExample (first 2 turns):")
for msg in all_conversations[0][:2]:
    print(f"  [{msg['role']}]: {msg['content'][:120]!r}")

Loading punctuation model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading spaCy model...
Loading conversations...
Loaded 186 conversations
Restoring punctuation and fixing capitalization...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Done.
Wrote 186 conversations → ../data/processed/daicwoz_finetune.jsonl

Example (first 2 turns):
  [assistant]: "Hi, I'm Ellie. Thanks for coming in today. I was created to talk to people in a safe and secure environment. Think of me"
  [user]: 'Good.'


## ESConv Preprocessing

Convert ESConv emotional support conversations to the same chat JSONL format.
`supporter` → `assistant` (Ellie), `seeker` → `user`.

In [2]:
import json
import warnings
from pathlib import Path

ESC_RAW_DIR = Path("../data/raw/esconv")
ESC_OUT_FILE = Path("../data/processed/esconv_finetune.jsonl")
ESC_OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = """\
You are a virtual clinical interviewer. Conduct a structured, empathetic \
mental health interview by asking open-ended questions and responds naturally \
to what the participant shares."""

def load_esconv_conversations(raw_dir: Path) -> list[list[dict]]:
    conversations = []
    for split_file in sorted(raw_dir.glob("*.jsonl")):
        with split_file.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                outer = json.loads(line)
                # Record is stored as a JSON string inside the "text" field
                inner = json.loads(outer["text"]) if isinstance(outer.get("text"), str) else outer
                dialog = inner.get("dialog", [])
                turns = []
                for utt in dialog:
                    speaker = utt.get("speaker", "")
                    text = utt.get("text", "").strip()
                    if not text:
                        continue
                    # sys = supporter (assistant/therapist), usr = seeker (user)
                    role = "assistant" if speaker == "sys" else "user"
                    if turns and turns[-1]["role"] == role:
                        turns[-1]["content"] += " " + text
                    else:
                        turns.append({"role": role, "content": text})
                if len(turns) >= 2:
                    conversations.append(turns)
    return conversations

print("Loading ESConv conversations...")
esconv_conversations = load_esconv_conversations(ESC_RAW_DIR)
print(f"Loaded {len(esconv_conversations)} conversations")
print(f"\nExample (first 2 turns):")
for msg in esconv_conversations[0][:2]:
    print(f"  [{msg['role']}]: {msg['content'][:120]!r}")

Loading ESConv conversations...
Loaded 1300 conversations

Example (first 2 turns):
  [assistant]: 'Hello. How are you today?'
  [user]: 'hi i am okay, a little bit sad though'


In [3]:
from dotenv import load_dotenv
load_dotenv("../.env")

import warnings
import re
import torch
import torch._dynamo  # pre-initialize before transformers to avoid circular import
import spacy
from transformers.pipelines import TokenClassificationPipeline
from deepmultilingualpunctuation import PunctuationModel

# Patch grouped_entities compatibility (safe to apply multiple times)
if not getattr(TokenClassificationPipeline, "_grouped_entities_patched", False):
    _orig_sanitize = TokenClassificationPipeline._sanitize_parameters

    def _patched_sanitize(self, **kwargs):
        if "grouped_entities" in kwargs:
            kwargs["aggregation_strategy"] = "simple" if kwargs.pop("grouped_entities") else "none"
        return _orig_sanitize(self, **kwargs)

    TokenClassificationPipeline._sanitize_parameters = _patched_sanitize
    TokenClassificationPipeline._grouped_entities_patched = True

SENTENCE_START_PATTERN = re.compile(r"([.!?]\s+)([a-z])")
PRONOUN_I_PATTERN = re.compile(r"\bi\b")
ELLIE_PATTERN = re.compile(r"\bellie\b", re.IGNORECASE)

def capitalize_sentences(text):
    text = text[:1].upper() + text[1:] if text else text
    return SENTENCE_START_PATTERN.sub(lambda m: m.group(1) + m.group(2).upper(), text)

def fix_pronoun_i(text):
    return PRONOUN_I_PATTERN.sub("I", text)

def fix_ellie(text):
    return ELLIE_PATTERN.sub("Ellie", text)

def capitalize_proper_nouns(text, nlp):
    doc = nlp(text)
    return "".join(
        (token.text if token.text.isupper() else token.text.capitalize()) + token.whitespace_
        if token.pos_ == "PROPN"
        else token.text_with_ws
        for token in doc
    )

print("Loading punctuation model...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    punct_model = PunctuationModel()

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

coords, texts = [], []
for c_idx, turns in enumerate(esconv_conversations):
    for t_idx, turn in enumerate(turns):
        coords.append((c_idx, t_idx))
        texts.append(turn["content"])

print(f"Processing {len(texts)} turns with punctuation model...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    restored_texts = [punct_model.restore_punctuation(t) for t in texts]

print("Applying capitalization fixes...")
for (c_idx, t_idx), restored in zip(coords, restored_texts):
    text = capitalize_sentences(restored)
    text = fix_pronoun_i(text)
    text = capitalize_proper_nouns(text, nlp)
    text = fix_ellie(text)
    esconv_conversations[c_idx][t_idx]["content"] = text
print("Done.")

records_written = 0
with ESC_OUT_FILE.open("w", encoding="utf-8") as out_f:
    for turns in esconv_conversations:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + turns
        out_f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        records_written += 1

print(f"Wrote {records_written} conversations → {ESC_OUT_FILE}")

Loading punctuation model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading spaCy model...
Processing 30392 turns with punctuation model...


KeyboardInterrupt: 

## AnnoMI Preprocessing

Convert AnnoMI motivational-interviewing transcripts to chat JSONL format.  
`therapist` → `assistant`, `client` → `user`.  
Only high-quality MI conversations are included by default (set `MI_QUALITY_FILTER = None` to include all).


## RealCBT Preprocessing

Convert RealCBT cognitive behavioral therapy transcripts to chat JSONL format.
`Counselor` → `assistant`, `Client` → `user`.
No punctuation or capitalization fixes needed — transcripts are already well-formatted.

In [1]:
import json
import re
from pathlib import Path

REALCBT_RAW_DIR = Path("../data/raw/realcbt")
REALCBT_OUT_FILE = Path("../data/processed/realcbt_finetune.jsonl")
REALCBT_OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = """\
You are a virtual clinical interviewer. Conduct a structured, empathetic \
mental health interview by asking open-ended questions and responds naturally \
to what the participant shares."""

# Matches speaker labels like "Counselor：" or "Client：" (full-width colon)
SPEAKER_PATTERN = re.compile(r"^(Counselor|Client)\s*[：:]", re.MULTILINE)

def load_realcbt_conversation(txt_path: Path) -> list[dict]:
    """Parse a RealCBT transcript file into a list of role/content dicts."""
    text = txt_path.read_text(encoding="utf-8")

    # Split on speaker labels, keeping the label
    parts = SPEAKER_PATTERN.split(text)
    # parts[0] is any text before the first speaker label (usually empty)
    # Then alternating: speaker_name, content, speaker_name, content, ...

    turns = []
    i = 1  # skip leading text
    while i + 1 < len(parts):
        speaker = parts[i].strip()
        content = parts[i + 1].strip()
        # Normalize whitespace (newlines within a turn become spaces)
        content = " ".join(content.split())
        i += 2
        if not content:
            continue
        role = "assistant" if speaker == "Counselor" else "user"
        if turns and turns[-1]["role"] == role:
            turns[-1]["content"] += " " + content
        else:
            turns.append({"role": role, "content": content})
    return turns

print("Loading RealCBT conversations...")
realcbt_conversations = []
for txt_path in sorted(REALCBT_RAW_DIR.glob("*.txt"), key=lambda p: int(re.search(r"\d+", p.stem).group())):
    turns = load_realcbt_conversation(txt_path)
    if len(turns) >= 2:
        realcbt_conversations.append(turns)

print(f"Loaded {len(realcbt_conversations)} conversations")

# Write JSONL
records_written = 0
with REALCBT_OUT_FILE.open("w", encoding="utf-8") as out_f:
    for turns in realcbt_conversations:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + turns
        out_f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        records_written += 1

print(f"Wrote {records_written} conversations → {REALCBT_OUT_FILE}")
print(f"\nExample (first 2 turns):")
for msg in realcbt_conversations[0][:2]:
    print(f"  [{msg['role']}]: {msg['content'][:120]!r}")

Loading RealCBT conversations...
Loaded 76 conversations
Wrote 76 conversations → ../data/processed/realcbt_finetune.jsonl

Example (first 2 turns):
  [assistant]: 'Hello, Aspen. How you doing today?'
  [user]: "I'm doing well, thank you. How about yourself?"


In [2]:
import csv
import json
from pathlib import Path

ANNOMI_RAW_FILE = Path("../data/raw/annomi/dataset.csv")
ANNOMI_OUT_FILE = Path("../data/processed/annomi_finetune.jsonl")
ANNOMI_OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# Set to "high", "low", or None (include both)
MI_QUALITY_FILTER = "high"

SYSTEM_PROMPT = """\
You are a virtual clinical interviewer. Conduct a structured, empathetic \
mental health interview by asking open-ended questions and responds naturally \
to what the participant shares."""

def load_annomi_conversations(csv_path: Path, mi_quality: str | None = "high") -> list[list[dict]]:
    # Group rows by transcript_id, preserving utterance_id order
    transcripts: dict[int, list[dict]] = {}
    with csv_path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if mi_quality and row["mi_quality"].strip().lower() != mi_quality.lower():
                continue
            tid = int(row["transcript_id"])
            transcripts.setdefault(tid, []).append(row)

    conversations = []
    for tid in sorted(transcripts):
        rows = sorted(transcripts[tid], key=lambda r: int(r["utterance_id"]))
        turns: list[dict] = []
        for row in rows:
            text = row["utterance_text"].strip()
            if not text:
                continue
            role = "assistant" if row["interlocutor"].strip().lower() == "therapist" else "user"
            if turns and turns[-1]["role"] == role:
                turns[-1]["content"] += " " + text
            else:
                turns.append({"role": role, "content": text})
        if len(turns) >= 2:
            conversations.append(turns)
    return conversations

print(f"Loading AnnoMI conversations (mi_quality={MI_QUALITY_FILTER!r})...")
annomi_conversations = load_annomi_conversations(ANNOMI_RAW_FILE, mi_quality=MI_QUALITY_FILTER)
print(f"Loaded {len(annomi_conversations)} conversations")
print(f"\nExample (first 2 turns):")
for msg in annomi_conversations[0][:2]:
    print(f"  [{msg['role']}]: {msg['content'][:120]!r}")

Loading AnnoMI conversations (mi_quality='high')...
Loaded 110 conversations

Example (first 2 turns):
  [assistant]: 'Thanks for filling it out. We give this form to everyone once a year regardless of why they come in. It helps us provide'
  [user]: 'Sure.'


In [5]:
import json

records_written = 0
with ANNOMI_OUT_FILE.open("w", encoding="utf-8") as out_f:
    for turns in annomi_conversations:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + turns
        out_f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        records_written += 1

print(f"Wrote {records_written} conversations → {ANNOMI_OUT_FILE}")

Wrote 110 conversations → ../data/processed/annomi_finetune.jsonl


## Merge Datasets

In [1]:
import random
from pathlib import Path

MERGED_OUT = Path("../data/processed/combined_finetune.jsonl")
TRAIN_OUT  = Path("../data/processed/combined_train.jsonl")
VAL_OUT    = Path("../data/processed/combined_val.jsonl")

daicwoz_lines = Path("../data/processed/daicwoz_finetune.jsonl").read_text(encoding="utf-8").splitlines()
esconv_lines  = Path("../data/processed/esconv_finetune.jsonl").read_text(encoding="utf-8").splitlines()
annomi_lines  = Path("../data/processed/annomi_finetune.jsonl").read_text(encoding="utf-8").splitlines()
realcbt_lines = Path("../data/processed/realcbt_finetune.jsonl").read_text(encoding="utf-8").splitlines()

all_lines = [l for l in annomi_lines + realcbt_lines if l.strip()]

# Save full combined file (unchanged)
with MERGED_OUT.open("w", encoding="utf-8") as f:
    f.write("\n".join(all_lines) + "\n")

# ── 90/10 train/val split ──────────────────────────────────────────────────
random.seed(42)
indices = list(range(len(all_lines)))
random.shuffle(indices)
split_idx = int(len(indices) * 0.9)
train_idx, val_idx = indices[:split_idx], indices[split_idx:]

train_lines = [all_lines[i] for i in train_idx]
val_lines   = [all_lines[i] for i in val_idx]

with TRAIN_OUT.open("w", encoding="utf-8") as f:
    f.write("\n".join(train_lines) + "\n")
with VAL_OUT.open("w", encoding="utf-8") as f:
    f.write("\n".join(val_lines) + "\n")

print(f"DAIC-WOZ : {len(daicwoz_lines)} conversations")
print(f"ESConv   : {len(esconv_lines)} conversations")
print(f"AnnoMI   : {len(annomi_lines)} conversations")
print(f"Combined : {len(all_lines)} conversations → {MERGED_OUT}")
print(f"Train    : {len(train_lines)} conversations → {TRAIN_OUT}")
print(f"Val      : {len(val_lines)} conversations → {VAL_OUT}")

DAIC-WOZ : 186 conversations
ESConv   : 1300 conversations
AnnoMI   : 110 conversations
Combined : 186 conversations → ../data/processed/combined_finetune.jsonl
Train    : 167 conversations → ../data/processed/combined_train.jsonl
Val      : 19 conversations → ../data/processed/combined_val.jsonl


## Chunk Long Conversations

Split conversations that exceed `MAX_SEQ_LENGTH` tokens into multiple training examples:

- **Chunk 1**: system prompt + complete turns until hitting the token limit.
- **Chunk 2+**: system prompt with a DeepSeek-generated summary of all prior turns appended, then complete turns until hitting the limit again.

This ensures no tokens are lost — every turn appears in at least one training example.

In [2]:
import os
import json
from pathlib import Path
from openai import OpenAI
from transformers import AutoTokenizer

from dotenv import load_dotenv
load_dotenv("../.env")


# ── Config ─────────────────────────────────────────────────────────────────
MODEL_NAME     = "Qwen/Qwen3.5-4B"
MAX_SEQ_LENGTH = 4096
SUMMARY_MODEL  = "deepseek-v4-flash"

TRAIN_PATH     = Path("../data/processed/combined_train.jsonl")
VAL_PATH       = Path("../data/processed/combined_val.jsonl")
TRAIN_OUT_PATH = Path("../data/processed/combined_train_chunked.jsonl")
VAL_OUT_PATH   = Path("../data/processed/combined_val_chunked.jsonl")

# ── Load tokenizer for token counting ──────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# ── DeepSeek client ────────────────────────────────────────────────────────
ds = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

USER_START_MSG = {"role": "user", "content": "Start the session."}

def ensure_user_first(messages):
    """If the first non-system turn is assistant, prepend a user message."""
    if len(messages) > 1 and messages[0]["role"] == "system" and messages[1]["role"] == "assistant":
        return [messages[0], USER_START_MSG] + messages[1:]
    return messages

def count_tokens(messages):
    """Count tokens for a message list using the model's chat template."""
    messages = ensure_user_first(messages)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return len(tokenizer.encode(text))

def summarize_previous_turns(turns):
    """Ask DeepSeek for a concise (≤ 1000 word) summary of prior conversation turns."""
    conversation_text = "\n".join(
        f"{'Therapist' if m['role'] == 'assistant' else 'Patient'}: {m['content']}"
        for m in turns
    )
    resp = ds.chat.completions.create(
        model=SUMMARY_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Summarize the following therapy conversation concisely in at most "
                    "1000 words. Focus on key topics discussed, patient concerns, "
                    "emotional state, and therapeutic progress. Output only the summary."
                ),
            },
            {"role": "user", "content": conversation_text},
        ],
        max_tokens=1500,
        temperature=0.3,
    )
    return resp.choices[0].message.content.strip()

def chunk_conversation(messages):
    """Split one conversation into chunks each ≤ MAX_SEQ_LENGTH tokens.

    Chunk 1 : system prompt  +  complete turns until hitting the limit.
    Chunk 2+: system prompt (with summary of ALL prior turns appended)
              +  complete turns until hitting the limit.
    """
    system_msg = messages[0]
    turns = messages[1:]

    chunks = []
    turn_idx = 0
    previous_turns = []  # accumulates every turn consumed so far

    while turn_idx < len(turns):
        # Build the system message for this chunk
        if not chunks:
            sys_msg = system_msg
        else:
            summary = summarize_previous_turns(previous_turns)
            sys_msg = {
                "role": "system",
                "content": (
                    system_msg["content"]
                    + "\n\n[Summary of conversation so far]\n"
                    + summary
                ),
            }

        chunk_msgs = [sys_msg]

        # Greedily add complete turns while under the token budget
        while turn_idx < len(turns):
            candidate = chunk_msgs + [turns[turn_idx]]
            if count_tokens(candidate) > MAX_SEQ_LENGTH and len(chunk_msgs) > 1:
                break  # would exceed limit; stop (but always keep ≥1 turn)
            chunk_msgs.append(turns[turn_idx])
            turn_idx += 1

        previous_turns.extend(chunk_msgs[1:])  # record turns for future summaries
        chunks.append({"messages": ensure_user_first(chunk_msgs)})

    return chunks

def process_jsonl(in_path, out_path):
    """Read a JSONL file, chunk long conversations, write to a new file."""
    records = []
    with open(in_path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    out = []
    needs_chunking = 0
    for i, rec in enumerate(records):
        msgs = ensure_user_first(rec["messages"])
        if count_tokens(msgs) <= MAX_SEQ_LENGTH:
            out.append({"messages": msgs})
        else:
            needs_chunking += 1
            out.extend(chunk_conversation(msgs))
        if (i + 1) % 100 == 0 or (i + 1) == len(records):
            print(f"  {i+1}/{len(records)} conversations processed "
                  f"({needs_chunking} chunked → {len(out)} total records)")

    with out_path.open("w", encoding="utf-8") as f:
        for rec in out:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    return out, needs_chunking

# ── Process train set ──────────────────────────────────────────────────────
print("Processing train set...")
train_records, train_chunked = process_jsonl(TRAIN_PATH, TRAIN_OUT_PATH)
print(f"  → {len(train_records)} records written to {TRAIN_OUT_PATH} "
      f"({train_chunked} conversations were chunked)\n")

# ── Process val set ────────────────────────────────────────────────────────
print("Processing val set...")
val_records, val_chunked = process_jsonl(VAL_PATH, VAL_OUT_PATH)

Processing train set...
  100/167 conversations processed (18 chunked → 126 total records)
  167/167 conversations processed (31 chunked → 208 total records)
  → 208 records written to ../data/processed/combined_train_chunked.jsonl (31 conversations were chunked)

Processing val set...
  19/19 conversations processed (4 chunked → 23 total records)
